# Training v2 — reproducible orchestrator for the 2×3 ablation

This notebook reproduces the full UAV face-recognition training + evaluation pipeline by calling standalone scripts in `scripts/`. The scripts skip work if their output checkpoint already exists (pass `--force` to re-run), so you can interrupt and resume.

## Pipeline structure

All Phase 2 runs share the v2 hyperparameters (Adam, batch 32, backbone lr `1e-6`, head/ArcFace lr `1e-4`, ArcFace `s=32 m=0.3`, 4 layers unfrozen) so the ablation isolates the variables of interest.

**Phase 1** (used as the backbone for `both_phases_*` configs):

| script | output | purpose |
|---|---|---|
| `generate_synthetic_identities_faceid.py` | `synthetic_identities/identity_001..030/` | 30 synthetic identities × 40 images each via IP-Adapter FaceID |
| `phase1_original.py` | `checkpoints/checkpoints/best_model.pth` | Phase 1 on VGGFace2 (no synth), s=64 m=0.5 |
| `phase1_with_synth.py` | `checkpoints/checkpoints/best_model_synth.pth` | Phase 1 on VGGFace2 + 30 synth ids |

**Phase 2** (the 4 ablation configs):

| script | output checkpoint | config name |
|---|---|---|
| `phase2_only_original.py` | `best_drone_model_phase2only.pth` | `phase2_only_original` |
| `phase2_only_synth.py` | `best_drone_model_phase2only_synth.pth` | `phase2_only_synth` |
| `both_phases_original.py` | `best_drone_model_v2.pth` | `both_phases_original` |
| `phase2_from_synth.py` | `best_drone_model_synth_v2.pth` | `both_phases_synth` |

**Evaluation:** `run_ablation_eval.py` runs leave-one-out cosine-centroid Rank-1 over all 7 configurations (raw_baseline + 6 trained) and writes `results/two_phase_ablation.csv`. The downstream `analysis_extensions.ipynb` regenerates t-SNE / per-identity / failure-case figures from the headline `both_phases_original` checkpoint.

In [ ]:
import os, sys, subprocess, time
from pathlib import Path

PROJECT_ROOT = Path('/home/buthaina.almulla/Documents/CV7502')
os.chdir(PROJECT_ROOT)
SCRIPTS = PROJECT_ROOT / 'scripts'
VENV_PYTHON = PROJECT_ROOT / 'venv' / 'bin' / 'python'
PYTHON = str(VENV_PYTHON) if VENV_PYTHON.exists() else sys.executable

def run_script(name, *args, force=False):
    """Invoke a script in scripts/ as a subprocess and stream its stdout/stderr live.
    Returns the return code. Each script handles its own idempotency."""
    cmd = [PYTHON, str(SCRIPTS / name), *args]
    if force and '--force' not in cmd:
        cmd.append('--force')
    print(f'\n>>> {" ".join(cmd)}', flush=True)
    t0 = time.time()
    rc = subprocess.run(cmd, cwd=str(PROJECT_ROOT)).returncode
    print(f'<<< {name} exited {rc} in {(time.time()-t0)/60:.1f} min', flush=True)
    if rc != 0:
        raise RuntimeError(f'{name} failed with rc={rc}')
    return rc

print(f'project root: {PROJECT_ROOT}')
print(f'python: {PYTHON}')

## Phase 1 (backbone adaptation on VGGFace2)

Each cell skips if its checkpoint already exists. Times below are rough estimates on a single GPU.

1. Generate 30 synthetic identities via IP-Adapter FaceID (~30 min)
2. Phase 1 on VGGFace2, no synth (~3-4 hr) → `best_model.pth`
3. Phase 1 on VGGFace2 + 30 synth ids (~3-4 hr) → `best_model_synth.pth`

In [ ]:
# Step 1: synthetic identity generation. Idempotent (skips per-identity if dir already exists).
run_script('generate_synthetic_identities_faceid.py')

In [ ]:
# Step 2: Phase 1 on VGGFace2 (no synth). Skips if best_model.pth exists.
run_script('phase1_original.py')

In [ ]:
# Step 3: Phase 1 on VGGFace2 + synth ids. Skips if best_model_synth.pth exists.
run_script('phase1_with_synth.py')

## Phase 2 (DroneFace fine-tuning) — the 4 ablation runs

All four use identical Phase 2 hyperparameters (v2 LRs, ArcFace `s=32 m=0.3`). The only differences:

| | from raw VGGFace2 | from Phase 1 (no synth) | from Phase 1 (with synth) |
|---|---|---|---|
| **DroneFace 8 ids only** | `phase2_only_original` | `both_phases_original` | — |
| **DroneFace 8 ids + 30 synth ids** | `phase2_only_synth` | — | `both_phases_synth` |

In [ ]:
run_script('phase2_only_original.py')

In [ ]:
run_script('phase2_only_synth.py')

In [ ]:
run_script('both_phases_original.py')

In [ ]:
run_script('phase2_from_synth.py')

## Evaluation: ablation CSV + analysis figures

1. `run_ablation_eval.py` evaluates all 7 configurations under leave-one-out cosine-centroid Rank-1 and writes `results/two_phase_ablation.csv`. It also refreshes `results/confusion_matrix.npy` and `results/embeddings/two_phase_droneface*.npy` from the headline `both_phases_original` checkpoint.
2. The next cell recomputes the `Ours (two-phase)` rows in `results/benchmarks/{droneface_main, per_distance, per_gender}.csv` from the freshly regenerated embeddings.
3. `analysis_extensions.ipynb` is executed via `nbconvert` to produce `results/analysis/{tsne_comparison, confusion_matrix_full, roc_curve_full, per_identity_accuracy, worst_failure_cases}.png`.

In [ ]:
run_script('run_ablation_eval.py')
print('\n--- results/two_phase_ablation.csv ---')
print((PROJECT_ROOT / 'results/two_phase_ablation.csv').read_text())

In [ ]:
import csv, numpy as np
import torch, torch.nn.functional as F
from collections import defaultdict

EMB = np.load(PROJECT_ROOT / 'results/embeddings/two_phase_droneface.npy')
LBL = np.load(PROJECT_ROOT / 'results/embeddings/two_phase_droneface_labels.npy')
TRAIN, VAL, TEST = list('ABCDEFGH'), ['I'], ['J', 'K']

files, idents = [], []
for sp, ids in [('train', TRAIN), ('validation', VAL), ('test', TEST)]:
    for ident in ids:
        folder = PROJECT_ROOT / 'datasets/droneface/split' / sp / ident
        for fn in sorted(os.listdir(folder)):
            if fn.lower().endswith(('.jpg', '.jpeg', '.png')):
                files.append(fn); idents.append(ident)
assert len(files) == EMB.shape[0]

HEIGHT_MAP = {'0': 1.5, '3': 3.0, '4': 4.0, '5': 5.0}
MALE = set('abcegjk'); FEMALE = set('dfhi')
def parse(fn):
    base = fn.lower().rsplit('.', 1)[0]; p = base.split('_')
    if len(p) < 5 or len(p[0]) != 1: return None, None, None
    h = HEIGHT_MAP.get(p[2])
    try: d = 17 - int(p[4])/2
    except ValueError: d = None
    g = 'male' if p[0] in MALE else ('female' if p[0] in FEMALE else None)
    return h, d, g

emb_t = torch.from_numpy(EMB).float()
unique = sorted(set(idents))
counts = {i: idents.count(i) for i in unique}
lbl_arr = np.array(idents)
sums = {i: emb_t[lbl_arr == i].sum(dim=0) for i in unique}
cents = torch.stack([sums[i]/counts[i] for i in unique]); cl = np.array(unique)
correct = np.zeros(len(emb_t), dtype=bool); r5_correct = 0
for i in range(len(emb_t)):
    ident = idents[i]
    adj = (sums[ident] - emb_t[i]) / (counts[ident] - 1)
    c = cents.clone(); c[unique.index(ident)] = adj
    cn = F.normalize(c, p=2, dim=1)
    q = F.normalize(emb_t[i].unsqueeze(0), p=2, dim=1)
    s = (cn @ q.T).squeeze().numpy(); o = np.argsort(-s)
    correct[i] = (cl[o[0]] == ident)
    if ident in cl[o[:5]]: r5_correct += 1
overall_r1 = correct.mean() * 100
overall_r5 = 100.0 * r5_correct / len(emb_t)
print(f'overall R1={overall_r1:.2f}  R5={overall_r5:.2f}')

dist = defaultdict(lambda: [0, 0])
for i, fn in enumerate(files):
    _, d, _ = parse(fn)
    if d is None: continue
    k = int(round(d/2)*2)
    dist[k][0] += int(correct[i]); dist[k][1] += 1
ours_dist = {f'{k}m': 100.0*v[0]/v[1] for k, v in sorted(dist.items())}

gender = defaultdict(lambda: [0, 0])
for i, fn in enumerate(files):
    _, _, g = parse(fn)
    if g is None: continue
    gender[g][0] += int(correct[i]); gender[g][1] += 1
fM = 100.0*gender['female'][0]/gender['female'][1]
mM = 100.0*gender['male'][0]/gender['male'][1]

def rewrite(path, new_row):
    with open(path) as f: rows = [r for r in csv.reader(f)]
    rows = [r for r in rows if not (r and r[0].startswith('Ours'))]
    rows.append(new_row)
    with open(path, 'w', newline='') as f: csv.writer(f).writerows(rows)

ordered = ['2m','4m','6m','8m','10m','12m','14m','16m']
rewrite(PROJECT_ROOT / 'results/benchmarks/per_distance.csv',
        ['Ours (two-phase)'] + [f'{ours_dist.get(k, float("nan")):.2f}' for k in ordered])
rewrite(PROJECT_ROOT / 'results/benchmarks/per_gender.csv',
        ['Ours (two-phase)', f'{fM:.2f}', f'{mM:.2f}', f'{fM-mM:+.2f}'])
rewrite(PROJECT_ROOT / 'results/benchmarks/droneface_main.csv',
        ['Ours (two-phase)', '27910327', '256', '107.985',
         'VGGFace2 + DroneFace (fine-tuned)', 'ArcFace (FT)',
         f'{overall_r1:.2f}', f'{overall_r5:.2f}'])
print('updated 3 benchmark CSVs')

In [ ]:
rc = subprocess.run([
    PYTHON, '-m', 'jupyter', 'nbconvert', '--to', 'notebook', '--execute',
    'notebooks/analysis_extensions.ipynb',
    '--output', 'analysis_extensions.ipynb',
    '--output-dir', 'notebooks',
], cwd=str(PROJECT_ROOT)).returncode
if rc != 0:
    raise RuntimeError(f'analysis_extensions.ipynb failed with rc={rc}')
print('analysis figures regenerated under results/analysis/')